# Chapter 17 — Volatility Smiles and Volatility Surfaces

教材 pp. 194–204；PDF 页序为教材页码加 11。



下列未明确标为教材例题的数据均为合成验证数据，不声称复现教材中的原始市场样本。

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from numpy.testing import assert_allclose
from volatility_surface_calculator import *

看涨看跌平价、隐波反解和 vega 的中心差分核对；pp.194–195,199。

In [2]:
option=EuropeanOption(100,.03,.01)
c=option.price(105,1,.2)
p=option.price(105,1,.2,-1)
assert_allclose(c-p,100*np.exp(-.01)-105*np.exp(-.03))
assert_allclose(option.implied_volatility(c,105,1,(.001,2)),.2)
numerical=(option.price(105,1,.200001)-option.price(105,1,.199999))/.000002
assert_allclose(option.greeks(105,1,.2).vega,numerical,rtol=1e-8)

教材表17.2及p.199：9个月、K/S=1.05 的隐波13.7%；1.5年、0.925 为14.525%。

In [3]:
grid=pd.DataFrame([[14.2,13,12,13.1,14.5],[14,13,12,13.1,14.2],
 [14.1,13.3,12.5,13.4,14.3],[14.7,14,13.5,14,14.8],
 [15,14.4,14,14.5,15.1],[14.8,14.6,14.4,14.7,15]],
 index=[1/12,.25,.5,1,2,5],columns=[.9,.95,1.,1.05,1.1])/100
surface=VolatilitySurface(grid)
queries=pd.DataFrame({'T':[.75,1.5],'m':[1.05,.925]})
assert_allclose(surface.evaluate(queries,'T','m').volatility,[.137,.14525])

隐含风险中性密度与已知对数正态密度比较；pp.203–204。

In [4]:
from scipy.stats import lognorm
option=EuropeanOption(100,.03)
strikes=np.linspace(60,140,801)
quotes=pd.DataFrame({'K':strikes,'C':option.price(strikes,1,.2)})
density=risk_neutral_density(quotes,'K','C',.03,1)
expected=lognorm.pdf(density.K,s=.2,scale=100*np.exp(.03-.5*.2**2))
assert_allclose(density.density,expected,atol=1e-7)

## 离散跳跃定价、MV delta及报价批量接口；pp.199–200

In [5]:
states=pd.DataFrame({'price':[42.,58.],'probability':[.5,.5]})
assert_allclose(jump_mixture_price(states,'price','probability',50,0,1),4)
assert_allclose(minimum_variance_delta(.5,20,-.001),.48)
option=EuropeanOption(100,.03)
quotes=option.quotes(pd.DataFrame({'K':[100.],'T':[1.],'sigma':[.2]}),'K','T','sigma')
assert_allclose(quotes.price,option.price(100,1,.2))